# check_submission_ver06

このノートブックは `annotations_gt_task_ver10.json` の先頭エントリについて、`tasks[0]` のみを実行して確認するための検証用です。

方針:
- `check_submission_ver05_re5.ipynb` の共通コードを流用
- GT の action をそのまま使用
- まずは先頭1件のみを対象

In [2]:
from pathlib import Path
import json
import sys
import logging

WORKSPACE = Path('/workspace')
VIDEO_DIR = WORKSPACE / 'data' / 'videos'
RULES_PATH = WORKSPACE / 'logs' / 'submit' / 'submission_ver05_json' / 'task_rules_ver05.json'
TASK_DECOMP_PATH = WORKSPACE / 'logs' / 'submit' / 'submission_ver05_json' / 'task_decomposition_ver05.json'
ANNOT_PATH = WORKSPACE / 'data' / 'annotations_gt_task_ver10.json'
OUT_DIR = WORKSPACE / 'logs' / 'notebook' / 'submission_ver06_tuning'
OUT_DIR.mkdir(parents=True, exist_ok=True)

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))
sys.path.append("/workspace/src")
from backup import task_rules_ver05_functions as task_rule_funcs
from src.utils.video_utility import load_video, write_video, show_before_after

rules_payload = json.loads(RULES_PATH.read_text(encoding='utf-8'))
rules = rules_payload['actions']
task_rows = json.loads(TASK_DECOMP_PATH.read_text(encoding='utf-8'))
instruction_by_video = {str(r.get('video_path', '')): str(r.get('instruction', '')) for r in task_rows}

logger = logging.getLogger('check_submission_ver06')
logger.setLevel(logging.INFO)
if not logger.handlers:
    h = logging.StreamHandler()
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter('[%(levelname)s] %(message)s'))
    logger.addHandler(h)

def run_action_core(video_path_in, video_path_out, action, params_override=None, instruction_override=None, show_compare=False):
    frames, fps, width, height = load_video(video_path_in)
    rule = dict(rules.get(action, {}))
    method = str(rule.get('method', 'identity'))

    params = dict(rule.get('params', {}))
    if params_override:
        params.update(params_override)

    if instruction_override is None:
        instruction = instruction_by_video.get(Path(video_path_in).name, '')
    else:
        instruction = instruction_override

    out_frames = task_rule_funcs.run_method(
        method=method,
        frames=frames,
        params=params,
        instruction=instruction,
        logger=logger,
    )
    write_video(video_path_out, out_frames, fps, width, height)
    if show_compare:
        show_before_after(video_path_in, video_path_out)
    return instruction, method, params

print('rules :', len(rules))
print('RULES_PATH:', RULES_PATH)
print('TASK_DECOMP_PATH:', TASK_DECOMP_PATH)
print('ANNOT_PATH:', ANNOT_PATH)

rules : 44
RULES_PATH: /workspace/logs/submit/submission_ver05_json/task_rules_ver05.json
TASK_DECOMP_PATH: /workspace/logs/submit/submission_ver05_json/task_decomposition_ver05.json
ANNOT_PATH: /workspace/data/annotations_gt_task_ver10.json


In [3]:
# annotations の先頭エントリ + tasks[0] を準備
SHOW_COMPARE = True
ANNOTATION_ID = 0
TASKS_ID = 0
ann_rows = json.loads(ANNOT_PATH.read_text(encoding='utf-8'))
if not ann_rows:
    raise ValueError('annotations is empty')

entry = dict(ann_rows[ANNOTATION_ID])
TARGET_VIDEO = str(entry.get('video_path', ''))
if not TARGET_VIDEO:
    raise ValueError('top annotation has empty video_path')

tasks = entry.get('tasks', [])
if not tasks:
    raise ValueError('top annotation has no tasks')

first_task = dict(tasks[TASKS_ID])
action = str(first_task.get('action', ''))
if not action:
    raise ValueError('tasks[0] action is empty')

video_in = VIDEO_DIR / TARGET_VIDEO
if not video_in.exists():
    raise FileNotFoundError(f'input video not found: {video_in}')

instruction = str(entry.get('instruction', ''))
SEQ_OUT_DIR = OUT_DIR / f'top{ANNOTATION_ID+1}_task{TASKS_ID+1}' / Path(TARGET_VIDEO).stem
SEQ_OUT_DIR.mkdir(parents=True, exist_ok=True)

print('target video:', TARGET_VIDEO)
print('class/subclass:', entry.get('class'), '/', entry.get('subclass'))
print('instruction:', instruction)
print('task[0] action:', action)
print('task[0] target:', first_task.get('target', ''))
print('task[0] constraints:', first_task.get('constraints', []))
print('task[0] params:', first_task.get('params', {}))

target video: wyzi9GNZFMU_0_0to121.mp4
class/subclass: Camera Motion Editing / Dolly in
instruction: Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.
task[0] action: dolly_in
task[0] target: man's face
task[0] constraints: ['smooth_motion']
task[0] params: {'motion_type': 'dolly_in', 'start_framing': 'medium_shot', 'end_framing': 'close_up'}


In [4]:
ann_rows

[{'video_path': 'wyzi9GNZFMU_0_0to121.mp4',
  'class': 'Camera Motion Editing',
  'subclass': 'Dolly in',
  'instruction': "Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.",
  'tasks': [{'action': 'dolly_in',
    'target': "man's face",
    'constraints': ['smooth_motion'],
    'params': {'motion_type': 'dolly_in',
     'start_framing': 'medium_shot',
     'end_framing': 'close_up'}},
   {'action': 'preserve_framing',
    'target': "man's face",
    'constraints': [],
    'params': {'position': 'center'}}]},
 {'video_path': '8rKYl1CdXCc_5_276to660_scene02.mp4',
  'class': 'Instance Motion Editing',
  'subclass': 'Human motion',
  'instruction': 'Modify the video so that the man raises his left hand to wave towards the camera in a friendly manner. Preserve his identity, the suit he is wearing, and the dark blue stage background. Ensure the motion is natural and temporally consistent, with n

In [ ]:
# 先頭エントリの tasks[0] を1つだけ実行
import importlib
importlib.reload(task_rule_funcs)

params = dict(first_task.get('params', {}))
params['_action'] = action
params['_constraints'] = list(first_task.get('constraints', []))
params['action'] = action
params['end_scale'] = 0.5
params['constraints'] = list(first_task.get('constraints', []))
if first_task.get('target', None) is not None:
    params['target'] = first_task.get('target', '')
params['video_path'] = TARGET_VIDEO

out_path = SEQ_OUT_DIR / f"{Path(TARGET_VIDEO).stem}__task01_{action}.mp4"
_instruction, method, used_params = run_action_core(
    video_in,
    out_path,
    action,
    params_override=params,
    instruction_override=instruction,
    show_compare=SHOW_COMPARE,
)

print('video      :', video_in.name)
print('action     :', action)
print('method     :', method)
print('out        :', out_path)
print('used params:', used_params)

dolly_in:object_zoom_in [wyzi9GNZFMU_0_0to121.mp4]:   0%|          | 0/120 [00:00<?, ?frame/s]

final text_encoder_type: bert-base-uncased


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15316.95it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
dolly_in:object_zoom_in [wyzi9GNZFMU_0_0to121.mp4]:  20%|██        | 24/120 [01:06<04:15,  2.66s/frame]